#### Dicionário dos dados:

- id_municipio =ID Município 

- pib	=	Produto Interno Bruto a preços correntes

- impostos_liquidos	=	Impostos, líquidos de subsídios, sobre produtos a preços correntes

- va_agropecuaria	=	Valor adicionado bruto a preços correntes da agropecuária

- va_industria =	Valor adicionado bruto a preços correntes da indústria

- va_servicos	=	Valor adicionado bruto a preços correntes dos serviços, exclusive administração, defesa, educação e saúde públicas e seguridade social

- va_adespss =	Valor adicionado bruto a preços correntes da administração, defesa, educação e saúde públicas e seguridade social

OBS: O Valor adicionado representa a contribuição do setor para o Produto Interno Bruto (PIB) de um país, ou seja, o valor que ele cria na produção de bens e serviços, expresso na moeda corrente do período, sem a aplicação de qualquer fator de correção ou deflator de preços. Ele é calculado pela diferença entre o valor da produção agropecuária e o consumo intermediário (ou insumos) utilizado na sua fabricação.

**Fonte:** https://basedosdados.org/dataset/fcf025ca-8b19-4131-8e2d-5ddb12492347?table=fbbbe77e-d234-4113-8af5-98724a956943

## Métricas consolidadas Base PIB por UF

In [0]:
CREATE OR REPLACE TABLE data_lake_hermes_ai.prata.metricas_pib_uf (
  WITH crescimento AS (
    SELECT
        sigla_uf,
        ano,
        pib,
        LAG(pib) OVER (PARTITION BY sigla_uf ORDER BY ano) AS pib_anterior,
        ROUND(
            (pib - LAG(pib) OVER (PARTITION BY sigla_uf ORDER BY ano)) 
            / LAG(pib) OVER (PARTITION BY sigla_uf ORDER BY ano) * 100, 2
        ) AS crescimento_percentual
    FROM data_lake_hermes_ai.bronze.br_ibge_pib_uf
),
media_uf AS (
    SELECT
        sigla_uf,
        AVG(pib) AS avg_pib,
        AVG(crescimento_percentual) AS avg_crescimento
    FROM crescimento
    GROUP BY sigla_uf
)
SELECT
    NTILE(4) OVER (ORDER BY avg_pib) AS quartil,
    sigla_uf,
    avg_pib,
    avg_crescimento
FROM media_uf
ORDER BY quartil, avg_pib
)

## Valor adicionado por setor Base PIB por UF

### Valor adicionado agropecuaria

In [0]:
CREATE OR REPLACE TABLE data_lake_hermes_ai.prata.pib_va_agropecuaria_uf (
  WITH crescimento AS (
    SELECT
        sigla_uf,
        ano,
        va_agropecuaria,
        LAG(va_agropecuaria) OVER (PARTITION BY sigla_uf ORDER BY ano) AS va_agropecuaria_anterior,
        ROUND(
            (va_agropecuaria - LAG(va_agropecuaria) OVER (PARTITION BY sigla_uf ORDER BY ano)) 
            / LAG(va_agropecuaria) OVER (PARTITION BY sigla_uf ORDER BY ano) * 100, 2
        ) AS percentual_va_agropecuaria
    FROM data_lake_hermes_ai.bronze.br_ibge_pib_uf
),
media_por_estado AS (
    SELECT
        sigla_uf,
        AVG(va_agropecuaria) AS avg_va_agropecuaria,
        AVG(percentual_va_agropecuaria) AS avg_crescimento
    FROM crescimento
    GROUP BY sigla_uf
)
SELECT
    NTILE(4) OVER (ORDER BY avg_va_agropecuaria) AS quartil,
    sigla_uf,
    avg_va_agropecuaria,
    avg_crescimento
FROM media_por_estado
ORDER BY quartil, avg_va_agropecuaria
)

### Valor adicionado industria

In [0]:
CREATE OR REPLACE TABLE data_lake_hermes_ai.prata.pib_va_industria_uf (
  WITH crescimento AS (
    SELECT
        sigla_uf,
        ano,
        va_industria,
        LAG(va_industria) OVER (PARTITION BY sigla_uf ORDER BY ano) AS va_industria_anterior,
        ROUND(
            (va_industria - LAG(va_industria) OVER (PARTITION BY sigla_uf ORDER BY ano)) 
            / LAG(va_industria) OVER (PARTITION BY sigla_uf ORDER BY ano) * 100, 2
        ) AS percentual_va_industria
    FROM data_lake_hermes_ai.bronze.br_ibge_pib_uf
),
media_uf AS (
    SELECT
        sigla_uf,
        AVG(va_industria) AS avg_va_industria,
        AVG(percentual_va_industria) AS avg_crescimento
    FROM crescimento
    GROUP BY sigla_uf
)
SELECT
    NTILE(4) OVER (ORDER BY avg_va_industria) AS quartil,
    sigla_uf,
    avg_va_industria,
    avg_crescimento
FROM media_uf
ORDER BY quartil, avg_va_industria)

### Valor adicionado Serviços

In [0]:
CREATE OR REPLACE TABLE data_lake_hermes_ai.prata.pib_va_servicos_uf (
  WITH crescimento AS (
    SELECT
        sigla_uf,
        ano,
        va_servicos,
        LAG(va_servicos) OVER (PARTITION BY sigla_uf ORDER BY ano) AS va_servicos_anterior,
        ROUND(
            (va_servicos - LAG(va_servicos) OVER (PARTITION BY sigla_uf ORDER BY ano)) 
            / LAG(va_servicos) OVER (PARTITION BY sigla_uf ORDER BY ano) * 100, 2
        ) AS percentual_va_servicos
    FROM data_lake_hermes_ai.bronze.br_ibge_pib_uf
),
media_uf AS (
    SELECT
        sigla_uf,
        AVG(va_servicos) AS avg_va_servicos,
        AVG(percentual_va_servicos) AS avg_crescimento
    FROM crescimento
    GROUP BY sigla_uf
)
SELECT
    NTILE(4) OVER (ORDER BY avg_va_servicos) AS quartil,
    sigla_uf,
    avg_va_servicos,
    avg_crescimento
FROM media_uf
ORDER BY quartil, avg_va_servicos)

### Valor adicionado Adespss

In [0]:
CREATE OR REPLACE TABLE data_lake_hermes_ai.prata.pib_va_adespss_uf ( 
WITH crescimento AS (
    SELECT
        sigla_uf,
        ano,
        va_adespss,
        LAG(va_adespss) OVER (PARTITION BY sigla_uf ORDER BY ano) AS va_adespss_anterior,
        ROUND(
            (va_adespss - LAG(va_adespss) OVER (PARTITION BY sigla_uf ORDER BY ano)) 
            / LAG(va_adespss) OVER (PARTITION BY sigla_uf ORDER BY ano) * 100, 2
        ) AS percentual_va_adespss
    FROM data_lake_hermes_ai.bronze.br_ibge_pib_uf
),
media_uf AS (
    SELECT
        sigla_uf,
        AVG(va_adespss) AS avg_va_adespss,
        AVG(percentual_va_adespss) AS avg_crescimento
    FROM crescimento
    GROUP BY sigla_uf
)
SELECT
    NTILE(4) OVER (ORDER BY avg_va_adespss) AS quartil,
    sigla_uf,
    avg_va_adespss,
    avg_crescimento
FROM media_uf
ORDER BY quartil, avg_va_adespss)